# 구위 모델 v4 : v3 − 구종(pitch_type) 변수 (2026-10-11, 사용자 지시로 웹 반영)

- 근거 : `outputs/hofnet_nopt/` 실험 — LGBM 은 구종을 빼도 검증 ρ(구위, whiff%) 0.719 → 0.717, 다음 시즌 예측 0.778 → 0.776 으로 거의 같고, 근거 중 '구종·투구손' 비중이 25% → 6% 로 줄어 물리량 중심으로 설명된다
- 학습 : 실험의 'LGBM 구종 제외' 와 같은 방법 (v3 하이퍼파라미터, 2025 학습셋 + 홀드아웃, 시드 42)
- 구종은 HAVAA 보정 계수 선택에만 쓰고 LGBM 입력에서는 뺀다 (input_features 에는 남고 기여는 0). 웹 마지막 요인 이름은 '투구손'
- Jupyter 커널 연결 문제로 같은 코드를 스크립트로 실행했다

In [ ]:
"""구위 모델 v4 = v3 에서 구종(pitch_type) 변수 제외 (stuff_pipeline_261011_v4.ipynb 와 같은 코드)
학습 : outputs/hofnet_nopt 실험의 'LGBM 구종 제외' 와 같은 방법 (v3 하이퍼파라미터, 2025 학습셋 + 홀드아웃, 시드 동일)"""
import os, json, time, joblib
import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy import stats
import stuff_pipeline as sp
import stuff_model as sm
import export_web_data as ew

T0 = time.time()
log = lambda *a: print(f'[{time.time() - T0:5.0f}s]', *a, flush=True)
CACHE = os.path.join('outputs', 'vaa_exp', 'cache')
v3 = sm.load_model(os.path.join('models', 'v3'))
v3_sc = joblib.load(os.path.join('models', 'v3', 'stage4_stuff_scaler.joblib'))
FEATS = [f for f in v3.features if f != 'pitch_type']
cols = sorted(set(v3.input_features) | {'game_date', 'waste', 'pitcher', 'description', 'type', 'estimated_woba_using_speedangle'})
fr = {y: pd.read_parquet(os.path.join(CACHE, f'frame_{y}.parquet'), columns=cols + (['y3', 'split'] if y == 2025 else []))
      for y in (2024, 2025, 2026)}
d = fr[2025]
valid_start = pd.Timestamp(pd.read_json(os.path.join(CACHE, 'meta.json'), typ='series')['valid_start'])
is_tr = (d['game_date'] < valid_start).to_numpy()
samp = d[d['y3'].notna()]
fit_rows = pd.concat([samp[samp['split'] == 'tr'], samp[samp['split'] == 'es']])
log(f'학습 표본 {len(fit_rows):,}구, 피처 {len(FEATS)}개 : {FEATS}')

lgbm = lgb.LGBMRegressor(**v3.lgbm.get_params()).fit(v3.model_frame(fit_rows[v3.input_features])[FEATS], fit_rows['y3'])
m = sm.AngleAdjustedStuffModel(lgbm, FEATS, v3.vaa_coef, vaa_default=v3.vaa_default, category_list=v3.categories)
log('input_features', m.input_features)
X = d[m.input_features]
p = m.predict(X)
for exact in (False, True):     # 기여 합 = 예측, 구종 열 기여 0
    s = X.sample(20000, random_state=42)
    c, b = m.explain(s, exact)
    assert np.abs(b + c.sum(1) - m.predict(s)).max() < 1e-9, exact
    assert np.abs(c[:, list(s.columns).index('pitch_type')]).max() == 0
scored_tr = is_tr & ~d['waste'].fillna(False).to_numpy().astype(bool) & ~sp.is_no_tracking(d).to_numpy()
scaler = sp.StuffScaler().fit(p[scored_tr])
log(f'2025 학습기간 점수 중앙값 {np.median(scaler.transform(p)[scored_tr]):.2f}')

def validity(x, s):
    x = x.assign(s=pd.Series(s, index=x.index).where(~x['waste'].fillna(False).astype(bool)),
                 swing=x['description'].isin(ew.SWING_DESC), whiff=x['description'].isin(sp.WHIFF_DESC))
    pp = x.groupby(['pitcher', 'pitch_type'], observed=True).agg(n=('swing', 'size'), s=('s', 'mean'), sw=('swing', 'sum'), wh=('whiff', 'sum')).query('n >= 100')
    return stats.spearmanr(pp['s'], pp['wh'] / pp['sw']).statistic

for y in (2025, 2026):
    x = fr[y]
    s4 = scaler.transform(m.predict(x[m.input_features]))
    s3 = v3_sc.transform(v3.predict(x[v3.input_features]))
    msg = f'{y} : v3 와 점수 상관 {np.corrcoef(s4, s3)[0, 1]:.4f}, 평균 |차이| {np.abs(s4 - s3).mean():.2f}점'
    if y == 2025:
        msg += f' · 검증 ρ(구위, whiff%) v3 {validity(x[~is_tr], s3[~is_tr]):.4f} / v4 {validity(x[~is_tr], s4[~is_tr]):.4f}'
    log(msg)
out = os.path.join('models', 'v4')
os.makedirs(out, exist_ok=True)
joblib.dump(m, os.path.join(out, sm.MODEL_FILE))
joblib.dump(scaler, os.path.join(out, 'stage4_stuff_scaler.joblib'), compress=3)
json.dump({'features': FEATS, 'input_features': m.input_features}, open(os.path.join(out, 'features.json'), 'w'), indent=1)
log('저장', out)
